In [2]:
import os, sys
sys.path.append("../..")
from utilities import plot_wellbore, RogiDataset
from pathlib import Path
import pandas as pd
from tqdm.notebook import tqdm
from plotly import express as px
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

data_path = "../../rogii-wellbore-geology-prediction"

# To Do
1. Data
    - Define what a sample should be.
    - Implement it
2. Model
3. Test


# TVT Prediction Architecture

## Problem Framing

The task is a **signal registration problem**. At each measured depth (MD) step along a lateral wellbore, we want to find where in the stratigraphic column the well sits — expressed as a TVT value from the typewell reference log.

The primary signal is the GR log measured along the lateral, which must be registered against the typewell GR log indexed by TVT. The typewell acts as a stratigraphic reference: it tells you what GR you should expect at each TVT position. Finding the TVT at each MD step is therefore equivalent to finding where in the typewell GR profile the lateral GR matches best.

## Target Variable

The model outputs a probability distribution (softmax) over TVT candidates at each MD step. The TVT candidates are not arbitrary bins — they are a direct lookup into the typewell, centered on the local TVT anchor:

$$\mathcal{T} = \{TVT_{\text{anchor}} - 50, \; TVT_{\text{anchor}} - 49.5, \; \ldots, \; TVT_{\text{anchor}} + 49.5, \; TVT_{\text{anchor}} + 50\}$$

Since the typewell is sampled at 0.5 TVT resolution, this gives exactly **200 candidates** per window. The candidates are a slice of the typewell array, so each candidate directly corresponds to a real typewell GR value — no discretization decisions required.

The model is trained with cross-entropy loss against the ground truth TVT bin index at each MD step. At inference, the argmax over the 200 candidates gives the predicted TVT, which is converted back to an absolute TVT value for RMSE scoring.

## Sliding Window Training

Rather than one pass per well covering the full lateral, the model is trained on sliding windows along the MD axis. For each window:

- The **local anchor** is the last known TVT just before the window starts.
- The **typewell slice** of 200 candidates is centered on that local anchor TVT.
- The **target** is delta TVT relative to the local anchor at each MD step within the window.

This means every window position in every well is a valid training example, not just the prediction zone — significantly multiplying the effective dataset size and ensuring the model sees consistent input structure regardless of where in the lateral the window falls.

At inference, multiple overlapping windows are run across the lateral, each producing TVT estimates for its region. Overlapping predictions are averaged, acting as test-time augmentation and smoothing out individual window errors.

## Architecture

The model has three components: two 1D CNN heads and a 2D CNN fusion network.

### Typewell Head

A 1D CNN that takes the 200-candidate typewell GR slice as input and outputs a dense feature map of shape $(C, 200)$. Each position encodes the local GR pattern around that TVT candidate in a learned feature space.

### Lateral Head

A 1D CNN with the same structure but separate weights, taking the GR window as input and outputting a feature map of shape $(C, N_{md})$. Each position encodes the local GR pattern around that MD step.

### Outer Product Grid

The two feature maps are combined into a 2D grid of shape $(2C, 200, N_{md})$ by broadcasting and concatenating:

$$\text{grid}[:, t, d] = [f_{\text{tw}}[:, t] \;||\; f_{\text{lat}}[:, d]]$$

Every cell $(t, d)$ holds the concatenated features for TVT candidate $t$ and MD step $d$ simultaneously. This exposes the full comparison between every TVT candidate and every MD position to the downstream network — the registration problem is laid out spatially in the grid.

### Fusion CNN

A 2D CNN that takes the $(2C, 200, N_{md})$ grid and outputs a $(1, 200, N_{md})$ score map. A softmax over the 200 TVT candidates at each MD step gives the final probability distribution. The 2D CNN learns to find the smooth path of high similarity through the grid — equivalent to learned dynamic time warping.

At 200 TVT candidates × $N_{md}$ window steps, the grid is small and memory-tractable, allowing generous batch sizes and feature channels.

## Preprocessing

- NaNs in the lateral GR are dropped and the log is resampled onto a uniform MD grid via linear interpolation.
- Both logs are normalized per well to zero mean and unit variance before input.
- The typewell GR slice of 200 candidates is extracted centered on the local anchor TVT for each window.

## Data Augmentation

- **Sliding windows**: every window position along every well is a training example, each with its own local anchor. This is the primary source of data augmentation.
- **Vertical shifts**: shift the local anchor TVT by a random constant offset, moving the typewell slice up or down. The GR signal is unchanged but the registration target shifts.
- **Stretching and compression**: warp the typewell TVT axis locally, simulating formation thickness variation between the typewell and the lateral. This teaches the model robustness to imperfect typewell-to-lateral correspondence.

In [3]:
class RogiDataset():
    def __init__(self, path=Path("../../rogii-wellbore-geology-prediction/train",)):
        self.path = path
        self.well_files={
            f.name.replace("__horizontal_well.csv",""):{"Well":f,"TypeWell":f.parent/f.name.replace("__horizontal_well.csv","__typewell.csv")}
            for f in self.path.glob("*__horizontal_well.csv")
            }
        self.keys = list(self.well_files.keys())
        self._index=0

    def __getitem__(self,key:str|int):
        if isinstance(key, int):
            key = self.keys[key]
        well_data, typewell_data = pd.read_csv(self.well_files[key]["Well"]),pd.read_csv(self.well_files[key]["TypeWell"])
        well_data, typewell_data = well_data.set_index("MD").sort_index(), typewell_data.set_index("TVT").sort_index()
        pred_start_idx = well_data.index[(~well_data.TVT_input.isna()).sum()]
        last_known_idx = well_data.index[~well_data.TVT_input.isna()][-1]
        well_data["Known"] = well_data.index<=last_known_idx
        return well_data, typewell_data

    def __len__(self):
        return len(self.keys)
    
    def __iter__(self):
        self._index=0
        return self

    def __next__(self):
        if self._index>= len(self):
            raise StopIteration
        val = self[self._index]
        self._index+=1
        return val
        

In [4]:
self = RogiDataset()


# Encoding the data
The model will take both the well and typewell logs.

## Well Log
The well log data, will be sent to the well log head of the model.
- It will be of shape (batch, channel, well_log_window_size)
    - well_log_window_size is a tunable hyperparameter. It should be large enough to capture enough GR variation to disambiguate the location from the typewell log.
- Each "sample" will be a window of gr data, anchored (at the start(left)) at any MD point along the wellbore.
- The spacing of the MD index will be 1 (the data default), and the gr log will have been interpolated to remove all nans.

## Typewell Log
The typewell log data, will be sent to the typewell log head of the model.
- shape: (batch, channel, typewell_log_window_size)
    - typewell_log_window_size is a tunable hyper parameter, this is the actual length of the window
    - The tvt span of the data within that window, will be set as another tunable hyper parameter: typewell_log_range
- Each sample, will be a window of the typewell log TVT, anchored in the middle (typewell_tvt_anchor), at the closest matching tvt
    - Since tvt index of the typewell is not always spread evenly, we need to reindex that window (bounds at )

In [228]:
class MatchingDataset(RogiDataset):
    def __init__(
            self, path=Path("../../rogii-wellbore-geology-prediction/train"), 
            well_wnd_length=100, typewell_wnd_length=100, typewell_delta_tvt=100
            ):
        super().__init__(path)
        self.well_wnd_length = well_wnd_length
        self.typewell_wnd_length = typewell_wnd_length
        self.typewell_delta_tvt = typewell_wnd_length

        well_num_idxs = []
        for i in tqdm(range(super().__len__())):
            w,t = super().__getitem__(i)
            well_num_idxs.append(w.shape[0]-(self.well_wnd_length-1))
        self.well_num_idxs = np.array(well_num_idxs, dtype=int)
        self.well_num_idxs_cumsum = self.well_num_idxs.cumsum()

    def __len__(self):
        return self.well_num_idxs.sum()
        
    def get_well_typewell_pair(self,idx):
        return super().__getitem__(idx)
    
    def __getitem__(self, key: str | int):
        selected_well = int(np.searchsorted(self.well_num_idxs_cumsum, key, side="right"))
        selected_well_idx = int(key-self.well_num_idxs[:selected_well].sum())
        well, typewell = self.get_well_typewell_pair(selected_well)
        well["GR"] = well.GR.interpolate()
        well_wnd = well.iloc[selected_well_idx:selected_well_idx+self.well_wnd_length]
        anchor_tvt = well_wnd.TVT.median()
        odd_typewell_wnd = typewell.loc[anchor_tvt-self.typewell_delta_tvt/2:anchor_tvt+self.typewell_delta_tvt/2]
        typewell_wnd = odd_typewell_wnd.GR.reindex(np.linspace(odd_typewell_wnd.index[0],odd_typewell_wnd.index[-1],self.typewell_wnd_length)).interpolate()
        return well_wnd, typewell_wnd
        
    
    
self = MatchingDataset()

  0%|          | 0/773 [00:00<?, ?it/s]

Dataset works, now have to add the md/tvt probability map the model will be trained to replicate.

In [229]:
w,t = self[0]

In [234]:
w.TVT.median()

np.float64(11286.38)

In [233]:
np.median(t.index)

np.float64(11286.2)